# Tesseract Baseline

Thin Colab/local entrypoint for OCR baseline execution.
Reusable logic stays in `src/`; this notebook only prepares runtime, uploads a sample, runs the pipeline, and exports artifacts.

Colab note: run from the top after pulling updates. The setup and extraction cells clear cached `src.*` modules so field candidates are generated from the latest project code.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('NFSE_REPO_URL', 'https://github.com/LuizCarlosGoulart/DolpII.git')


def default_repo_root():
    if IS_COLAB:
        return Path('/content/nfse-extractor')

    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

        nested = candidate / 'nfse-extractor'
        if (nested / 'src').is_dir() and (nested / 'notebooks').is_dir():
            return nested

    return cwd


REPO_ROOT = Path(os.environ.get('NFSE_REPO_ROOT', str(default_repo_root())))
PROJECT_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', str(REPO_ROOT / 'nfse-extractor' if IS_COLAB else REPO_ROOT)))

CONFIG = {
    'clone_or_pull': os.environ.get('NFSE_CLONE_OR_PULL', '1') == '1',
    'bootstrap_runtime': os.environ.get('NFSE_BOOTSTRAP_RUNTIME', '1') == '1',
    'mount_drive': os.environ.get('NFSE_MOUNT_DRIVE', '0') == '1',
    'sample_path': os.environ.get('NFSE_SAMPLE_PATH', ''),
    'language': os.environ.get('NFSE_TESSERACT_LANGUAGE', 'por'),
    'raw_output_path': os.environ.get('NFSE_TESSERACT_RAW_OUTPUT', '/content/tesseract_raw_artifacts.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_raw_artifacts.json')),
    'candidate_output_path': os.environ.get('NFSE_TESSERACT_CANDIDATE_OUTPUT', '/content/tesseract_field_candidates.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_field_candidates.json')),
    'preview_limit': int(os.environ.get('NFSE_PREVIEW_LIMIT', '80')),
    # Cache: skip OCR for documents whose raw artifact JSON already exists on disk.
    # Set NFSE_OCR_CACHE=0 to force re-extraction (e.g. after changing DPI or language).
    'use_ocr_cache': os.environ.get('NFSE_OCR_CACHE', '1') == '1',
    # Parallel workers for Tesseract (each call is a subprocess — thread-safe).
    # Auto-detects available CPUs; Colab typically has 2, local machines 4–8.
    'max_workers': int(os.environ.get('NFSE_MAX_WORKERS', str(min(4, os.cpu_count() or 2)))),
}


def reset_project_imports():
    """Force Python to import project modules from disk after git updates."""
    importlib.invalidate_caches()
    removed = 0
    for module_name in list(sys.modules):
        if module_name == 'src' or module_name.startswith('src.'):
            del sys.modules[module_name]
            removed += 1
    return removed


def project_git_head():
    git_cwd = PROJECT_ROOT if PROJECT_ROOT.exists() else REPO_ROOT
    result = subprocess.run(
        ['git', '-C', str(git_cwd), 'log', '-1', '--oneline'],
        check=False,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        return 'not a git checkout yet'
    return result.stdout.strip() or 'unknown'


print(f'IS_COLAB: {IS_COLAB}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'use_ocr_cache: {CONFIG["use_ocr_cache"]}')
print(f'max_workers: {CONFIG["max_workers"]}')

In [ ]:
if IS_COLAB and CONFIG['clone_or_pull']:
    if (REPO_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
    elif not REPO_ROOT.exists() or (REPO_ROOT.is_dir() and not any(REPO_ROOT.iterdir())):
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        raise FileExistsError(
            f'{REPO_ROOT} exists but is not a git checkout. Remove it or set NFSE_REPO_ROOT to another path.'
        )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {PROJECT_ROOT}. Check NFSE_PROJECT_ROOT or the cloned repository layout.'
    )

project_root_str = str(PROJECT_ROOT)
while project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')

if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(PROJECT_ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

removed_modules = reset_project_imports()
print(f'Runtime ready: {project_git_head()}')
print(f'Project module cache cleared: {removed_modules} module(s)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# File upload / discovery  ─  supports single OR multiple PDFs/images
# ─────────────────────────────────────────────────────────────────────────────
if IS_COLAB and not CONFIG['sample_path']:
    from google.colab import files

    print('Upload one or more PDF/image files (Ctrl+click to select multiple):')
    uploaded = files.upload()

    sample_paths_raw: list[Path] = []
    for _fname in uploaded:
        _candidate = Path('/content') / _fname
        if not _candidate.exists():
            _matches = (
                sorted(Path.cwd().glob(_fname))
                + sorted(Path('/content').rglob(_fname))
            )
            _candidate = _matches[-1] if _matches else _candidate
        sample_paths_raw.append(_candidate)
else:
    # NFSE_SAMPLE_PATH may be a comma-separated list  (e.g. 'a.pdf,b.pdf,c.pdf')
    _raw_value = CONFIG['sample_path']
    if _raw_value:
        sample_paths_raw = [
            Path(p.strip()).expanduser()
            for p in _raw_value.split(',')
            if p.strip()
        ]
    else:
        sample_paths_raw = []

if not sample_paths_raw:
    raise ValueError(
        'No files to process. '
        'Upload files in Colab or set NFSE_SAMPLE_PATH before running.'
    )

# Validate existence; warn and skip any missing paths
sample_paths: list[Path] = []
for _p in sample_paths_raw:
    if _p.exists():
        sample_paths.append(_p)
    else:
        print(f'[WARN] File not found, skipping: {_p}')

if not sample_paths:
    raise FileNotFoundError('None of the provided files exist.')

# Keep CONFIG['sample_path'] pointing to the first file (backward compat)
CONFIG['sample_path'] = str(sample_paths[0])
sample_path = sample_paths[0]   # single-path variable preserved

print(f'Files queued for processing: {len(sample_paths)}')
for _i, _p in enumerate(sample_paths, 1):
    print(f'  [{_i}] {_p.name}  ({_p.stat().st_size // 1024} KB)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Batch OCR extraction  ─  processes every file in sample_paths
# Cache:      if NFSE_OCR_CACHE=1 (default) and raw artifact JSON already
#             exists for a document, OCR is skipped entirely.
# Parallel:   documents are processed concurrently using ThreadPoolExecutor.
#             Tesseract spawns a subprocess per call → thread-safe.
# ─────────────────────────────────────────────────────────────────────────────
import gc
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter

removed_modules = reset_project_imports()
print(f'Using project commit: {project_git_head()}')
print(f'Project module cache cleared before extraction: {removed_modules} module(s)')
print(f'Parallel workers : {CONFIG["max_workers"]}')
print(f'OCR cache        : {"on" if CONFIG["use_ocr_cache"] else "off"}')
print()

from src.engines import TesseractExtractionAdapter
from src.export import load_extracted_elements_json, write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

normalization_module = importlib.import_module('src.normalization')
ConfigDrivenOutputNormalizer = getattr(normalization_module, 'ConfigDrivenOutputNormalizer', None)

# ── Shared resources  ─  instantiated ONCE, reused across all threads ─────────
adapter    = TesseractExtractionAdapter(language=CONFIG['language'])
normalizer = ConfigDrivenOutputNormalizer() if ConfigDrivenOutputNormalizer is not None else None

# ── Output path helpers ────────────────────────────────────────────────────────
_raw_base_dir = Path(CONFIG['raw_output_path']).parent
_batch_root   = _raw_base_dir / 'tesseract_batch'


def _output_paths(stem, single_file):
    """Return (raw_path, candidate_path) for one document."""
    if single_file:
        return Path(CONFIG['raw_output_path']), Path(CONFIG['candidate_output_path'])
    doc_dir = _batch_root / stem
    doc_dir.mkdir(parents=True, exist_ok=True)
    return (
        doc_dir / (stem + '_raw_artifacts.json'),
        doc_dir / (stem + '_field_candidates.json'),
    )


# ── Thread-safe print helper ───────────────────────────────────────────────────
_print_lock = threading.Lock()


def _tprint(*args, **kwargs):
    with _print_lock:
        print(*args, **kwargs)


# ── Per-document worker ────────────────────────────────────────────────────────
_is_single = len(sample_paths) == 1


def _process_one(file_idx: int, cur_path: Path) -> dict:
    """Extract, normalise and persist artifacts for a single document.

    Returns a result dict consumed by the batch summary below.
    Exceptions are caught and returned as status='error' so one failure
    does not abort the whole batch.
    """
    stem = cur_path.stem
    _t0 = perf_counter()

    try:
        raw_path, cand_path = _output_paths(stem, _is_single)
        doc = load_document(cur_path)

        # ── OCR (or cache hit) ─────────────────────────────────────────────
        cached = False
        if CONFIG['use_ocr_cache'] and raw_path.exists():
            try:
                doc_artifacts = load_extracted_elements_json(raw_path)
                cached = True
            except Exception:
                # Corrupted cache file → fall through to re-extract
                doc_artifacts = None

        if not cached:
            preprocessed = preprocess_document(doc)
            doc_artifacts = adapter.extract_preprocessed(preprocessed)
            del preprocessed
            gc.collect()
            write_extracted_elements_json(doc_artifacts, raw_path)

        # Infer page count without re-opening the file when loaded from cache
        page_count = (
            max((e.page_number or 1) for e in doc_artifacts)
            if doc_artifacts else 0
        )

        # ── Field normalisation ────────────────────────────────────────────
        doc_candidates = []
        if normalizer is not None:
            doc_candidates = normalizer.normalize(doc, doc_artifacts)
            cand_path.parent.mkdir(parents=True, exist_ok=True)
            cand_path.write_text(
                json.dumps(
                    [c.model_dump(mode='json') for c in doc_candidates],
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding='utf-8',
            )
        else:
            cand_path = None

        elapsed_ms = round((perf_counter() - _t0) * 1000, 1)
        cache_tag  = ' [cache]' if cached else ''
        _tprint(
            f'  [{file_idx}/{len(sample_paths)}] {cur_path.name}'
            f'  →  {len(doc_artifacts)} elements / {len(doc_candidates)} candidates'
            f'  ({elapsed_ms} ms){cache_tag}'
        )

        return {
            'index':          file_idx,
            'file':           cur_path.name,
            'status':         'ok',
            'cached':         cached,
            'document_id':    doc.document_id,
            'page_count':     page_count,
            'raw_elements':   len(doc_artifacts),
            'candidates':     len(doc_candidates),
            'raw_path':       str(raw_path),
            'candidate_path': str(cand_path) if cand_path else None,
            'elapsed_ms':     elapsed_ms,
            # Keep objects for the preview cells (only last-doc assignment is used)
            '_doc':           doc,
            '_artifacts':     doc_artifacts,
            '_candidates':    doc_candidates,
            '_raw_path':      raw_path,
            '_cand_path':     cand_path,
        }

    except Exception as exc:
        elapsed_ms = round((perf_counter() - _t0) * 1000, 1)
        _tprint(f'  [{file_idx}/{len(sample_paths)}] {cur_path.name}  →  ERROR: {exc}')
        traceback.print_exc()
        gc.collect()
        return {
            'index':      file_idx,
            'file':       cur_path.name,
            'status':     'error',
            'error':      str(exc),
            'elapsed_ms': elapsed_ms,
        }


# ── Parallel batch execution ───────────────────────────────────────────────────
batch_results: list[dict] = []

_t_batch = perf_counter()
with ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
    future_map = {
        executor.submit(_process_one, idx, path): idx
        for idx, path in enumerate(sample_paths, 1)
    }
    for future in as_completed(future_map):
        batch_results.append(future.result())

# Restore original file order for display / downstream cells
batch_results.sort(key=lambda r: r['index'])

_batch_elapsed = round((perf_counter() - _t_batch) * 1000, 1)

# ── Expose last successful document for preview cells ─────────────────────────
# (Set BEFORE checking for total failure so the variables always exist)
artifacts             = []
candidates            = []
document              = None
raw_output_path       = None
candidate_output_path = None

for _r in batch_results:
    if _r['status'] == 'ok':
        document              = _r['_doc']
        artifacts             = _r['_artifacts']
        candidates            = _r['_candidates']
        raw_output_path       = _r['_raw_path']
        candidate_output_path = _r['_cand_path']

print()
print('=' * 60)
_n_ok    = sum(1 for r in batch_results if r['status'] == 'ok')
_n_cache = sum(1 for r in batch_results if r.get('cached'))
_n_err   = sum(1 for r in batch_results if r['status'] == 'error')
print(
    f'Batch complete  {_n_ok} ok ({_n_cache} from cache) / '
    f'{_n_err} failed / {len(sample_paths)} total  '
    f'[{_batch_elapsed} ms total]'
)

if document is None:
    raise RuntimeError('Every document failed to process. See errors above.')

print()
print('Active document for preview cells:')
print(f'  document_id : {document.document_id}')
print(f'  media_type  : {document.media_type}')
print(f'  raw_output  : {raw_output_path}')
print(f'  candidates  : {candidate_output_path}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Batch results table  ─  one row per processed document
# ─────────────────────────────────────────────────────────────────────────────
_sep = '-' * 96
print(f'  {"#":<4} {"File":<35} {"Status":<8} {"C?":<4} {"Pages":<6} {"Elements":<10} {"Cands":<8} {"ms":>7}')
print(_sep)
for _r in batch_results:
    _fname = _r['file'][:34]
    if _r['status'] == 'ok':
        _c = '✓' if _r.get('cached') else ' '
        print(
            f'  {_r["index"]:<4} {_fname:<35} {"ok":<8} {_c:<4}'
            f'{_r["page_count"]:<6} {_r["raw_elements"]:<10} '
            f'{_r["candidates"]:<8} {_r["elapsed_ms"]:>7}'
        )
    else:
        _short_err = str(_r.get('error', '?'))[:40]
        print(
            f'  {_r["index"]:<4} {_fname:<35} {"FAILED":<8} {"":4}'
            f'{"--":<6} {"--":<10} {"--":<8} {_r["elapsed_ms"]:>7}   <- {_short_err}'
        )
print(_sep)
print('C? = loaded from OCR cache (no re-extraction)')
if len(sample_paths) > 1:
    print(f'Output root: {_batch_root}')
else:
    print(f'Output: {raw_output_path}')

In [ ]:
print('Raw OCR preview')
for item in artifacts[: CONFIG['preview_limit']]:
    print(
        f'{item.text!r} | conf={item.confidence} | page={item.page_number} | bbox={item.bounding_box}'
    )

if candidates:
    print('\nField candidate preview')
    for candidate in candidates[: CONFIG['preview_limit']]:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'section={candidate.metadata.get("section_name")} | '
            f'label={candidate.metadata.get("label_text")}'
        )


In [ ]:
from pathlib import Path
import src.normalization as normalization
import src.normalization.output_normalizer as output_normalizer

print("project commit:", project_git_head())
print("normalization file:", normalization.__file__)
print("output normalizer file:", output_normalizer.__file__)
print("has ConfigDrivenOutputNormalizer:", hasattr(normalization, "ConfigDrivenOutputNormalizer"))
print("candidates exists:", "candidates" in globals())
print("candidates len:", len(candidates) if "candidates" in globals() else None)
print("output_normalizer.py exists:", (PROJECT_ROOT / "src" / "normalization" / "output_normalizer.py").exists())


In [ ]:
important_fields = {
    "nfse_number",
    "issue_date",
    "verification_code",
    "provider_name",
    "provider_document",
    "provider_municipal_registration",
    "provider_address",
    "provider_email",
    "provider_uf",
    "provider_phone",
    "recipient_name",
    "recipient_document",
    "recipient_municipal_registration",
    "recipient_address",
    "recipient_email",
    "recipient_uf",
    "recipient_phone",
    "service_description",
    "service_code",
    "operation_nature",
    "service_city",
    "gross_amount",
    "deductions_amount",
    "taxable_amount",
    "iss_rate",
    "iss_amount",
    "iss_withheld_amount",
    "net_amount",
    "unconditional_discount",
    "conditional_discount",
    "pis_withheld_amount",
    "cofins_withheld_amount",
    "csll_withheld_amount",
    "inss_withheld_amount",
    "ir_withheld_amount",
    "other_retentions_amount",
}

for candidate in candidates:
    if candidate.field_name in important_fields:
        print(
            f"{candidate.field_name} = {candidate.value!r} | "
            f"conf={candidate.confidence} | "
            f"section={candidate.metadata.get('section_name')} | "
            f"label={candidate.metadata.get('label_text')} | "
            f"review={candidate.metadata.get('review_status')} | "
            f"reasons={candidate.metadata.get('review_reasons')} | "
            f"line={candidate.metadata.get('line_text')}"
        )

In [ ]:
from collections import Counter

review_counts = Counter(candidate.metadata.get('review_status', 'unknown') for candidate in candidates)
print('Review status summary')
for status, count in sorted(review_counts.items()):
    print(status, count)

needs_review = [c for c in candidates if c.metadata.get('manual_review_required')]
if needs_review:
    print()
    print('Needs review')
    for candidate in needs_review:
        print(
            f"{candidate.field_name} = {candidate.value!r} | "
            f"conf={candidate.confidence} | "
            f"reasons={candidate.metadata.get('review_reasons')} | "
            f"line={candidate.metadata.get('line_text')}"
        )
else:
    print()
    print('No candidate-level review flags.')


In [ ]:
from collections import Counter

counts = Counter(candidate.field_name for candidate in candidates)

for field_name, count in sorted(counts.items()):
    print(field_name, count)
